# 04 · Consultas

Responde a la pregunta del proyecto con las consultas de `sql/02_consultas.sql`.

**Pregunta:** ¿qué jugador ha metido más triples en proporción al tiempo jugado en la historia de la NBA?

**Hipótesis:** Stephen Curry es el jugador con más triples por cada 36 minutos jugados.

In [1]:
import os
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# lee usuario y contraseña del archivo .env de la raíz del repo
load_dotenv("../.env")

DB_NAME = os.getenv("DB_NAME", "nba_triples")

def crear_url(base_de_datos=None):
    return URL.create(
        "mysql+pymysql",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST", "localhost"),
        port=int(os.getenv("DB_PORT", 3306)),
        database=base_de_datos,
        query={"charset": "utf8mb4"},
    )

engine = create_engine(crear_url(DB_NAME))

def consulta(sql):
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

## 1. Ranking de triples por cada 36 minutos (mínimo 5.000 minutos)

Exigimos un mínimo de minutos para que no salga primero alguien que jugó tres partidos.

In [2]:
consulta("""
SELECT j.nombre,
       SUM(e.minutos) AS minutos,
       SUM(e.triples) AS triples,
       ROUND(SUM(e.triples) * 36 / SUM(e.minutos), 2) AS triples_por_36
FROM estadisticas e
JOIN jugadores j ON j.player_id = e.player_id
GROUP BY j.player_id, j.nombre
HAVING SUM(e.minutos) >= 5000
ORDER BY triples_por_36 DESC
LIMIT 10
""")

,nombre,minutos,triples,triples_por_36
0,Stephen Curry,36444.0,4270.0,4.22
1,Isaiah Joe,6985.0,768.0,3.96
2,Troy Daniels,5045.0,555.0,3.96
3,Duncan Robinson,13304.0,1444.0,3.91
4,Sam Hauser,6768.0,733.0,3.90
5,Malik Beasley,13739.0,1453.0,3.81
6,Buddy Hield,21083.0,2207.0,3.77
7,LaMelo Ball,9568.0,997.0,3.75
8,Dāvis Bertāns,8546.0,885.0,3.73
9,A.J. Green,5004.0,512.0,3.68


## 2. ¿Cambia el ranking con un mínimo de 20.000 minutos? (CTE + RANK)

Si el primero sigue siendo el mismo con mínimos más altos, la conclusión no depende del corte que elijamos.

In [3]:
consulta("""
WITH carrera AS (
    SELECT player_id,
           SUM(minutos) AS minutos,
           SUM(triples) * 36 / SUM(minutos) AS triples_por_36
    FROM estadisticas
    GROUP BY player_id
    HAVING SUM(minutos) >= 20000
)
SELECT RANK() OVER (ORDER BY c.triples_por_36 DESC) AS puesto,
       j.nombre, c.minutos, ROUND(c.triples_por_36, 2) AS triples_por_36
FROM carrera c
JOIN jugadores j ON j.player_id = c.player_id
ORDER BY puesto
LIMIT 5
""")

,puesto,nombre,minutos,triples_por_36
0,1,Stephen Curry,36444.0,4.22
1,2,Buddy Hield,21083.0,3.77
2,3,Klay Thompson,29432.0,3.57
3,4,Donovan Mitchell,20783.0,3.21
4,5,Damian Lillard,32759.0,3.10


## 3. Triples totales frente a triples por minuto

Los que más triples han metido en total no son necesariamente los que más meten por minuto.

In [4]:
consulta("""
SELECT j.nombre,
       SUM(e.triples) AS triples_totales,
       ROUND(SUM(e.triples) * 36 / SUM(e.minutos), 2) AS triples_por_36
FROM estadisticas e
JOIN jugadores j ON j.player_id = e.player_id
GROUP BY j.player_id, j.nombre
ORDER BY triples_totales DESC
LIMIT 10
""")

,nombre,triples_totales,triples_por_36
0,Stephen Curry,4270.0,4.22
1,James Harden,3405.0,2.87
2,Ray Allen,2973.0,2.31
3,Klay Thompson,2918.0,3.57
4,Damian Lillard,2820.0,3.10
5,LeBron James,2641.0,1.55
6,Reggie Miller,2560.0,1.94
7,Paul George,2465.0,2.79
8,Kyle Korver,2450.0,2.83
9,Kevin Durant,2388.0,1.94


## 4. Evolución del triple por décadas

Triples por cada 36 minutos y porcentaje de acierto de toda la liga.

In [5]:
consulta("""
SELECT (temporada DIV 10) * 10 AS decada,
       ROUND(SUM(triples) * 36 / SUM(minutos), 2) AS triples_por_36,
       ROUND(SUM(triples) / SUM(triples_intentados), 3) AS acierto
FROM estadisticas
GROUP BY decada
ORDER BY decada
""")

,decada,triples_por_36,acierto
0,1980,0.15,0.289
1,1990,0.59,0.347
2,2000,0.83,0.356
3,2010,1.23,0.356
4,2020,1.91,0.360


## 5. Posiciones de los 50 mejores tiradores por minuto

¿De qué posición son los mejores tiradores?

In [6]:
consulta("""
WITH top50 AS (
    SELECT player_id,
           SUM(triples) * 36 / SUM(minutos) AS triples_por_36
    FROM estadisticas
    GROUP BY player_id
    HAVING SUM(minutos) >= 5000
    ORDER BY triples_por_36 DESC
    LIMIT 50
)
SELECT j.posicion, COUNT(*) AS jugadores
FROM top50 t
JOIN jugadores j ON j.player_id = t.player_id
GROUP BY j.posicion
ORDER BY jugadores DESC
""")

,posicion,jugadores
0,G,32
1,F,10
2,G-F,6
3,F-G,1
4,F-C,1


## Conclusiones

- Stephen Curry es el primero en triples por cada 36 minutos con cualquiera de los mínimos de minutos que hemos probado. La hipótesis se confirma.
- Detrás de él el ranking cambia según el mínimo: con pocos minutos aparecen especialistas que jugaron poco; con muchos, tiradores con carreras largas.
- El triple ha crecido muchísimo desde los años 80, así que los jugadores recientes salen favorecidos.
- Son relaciones entre datos, no causas.